# 02_experimentation: Gaussian Process Model Development
## Comeback Analytics — Kernel Comparison & Cross-Validation

**Objective:** Build Gaussian Process regression models with different kernels, compare predictive performance, and validate assumptions.

**Expected outputs:**
- Fitted GP models (Matérn 5/2, RBF, Matérn 3/2)
- Cross-validation results and performance metrics
- Posterior uncertainty bands
- Model diagnostics and comparison plots
- Recommendation of best kernel for formalization phase


## 1. Setup & Load Cleaned Data


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

# Gaussian Process libraries
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, ConstantKernel as C
from sklearn.model_selection import cross_val_score, LeaveOneOut
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# Find project root dynamically
ROOT = Path().resolve()
while not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.append(str(ROOT))

from src.paths import MODELS_DIR, RESULTS_DIR, DATA_DIR

# Seed for reproducibility
np.random.seed(42)
 
# Styling
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
 

In [ ]:
# DATA LOADING
comeback_df = pd.read_csv(RESULTS_DIR / "comeback_clean.csv")


## 2. Prepare Data for Modeling


In [ ]:
# Prepare feature matrix and target
X = comeback_df[['time_elapsed_in_period']].values  # Predictor: time (seconds)
y = comeback_df['xGoal'].values   # Outcome: xGoal per shot

# Normalize time to [0, 1] for numerical stability
X_min, X_max = X.min(), X.max()
X_normalized = (X - X_min) / (X_max - X_min)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nX (time) stats:")
print(f"  Raw: min={X.min():.0f}, max={X.max():.0f}")
print(f"  Normalized: min={X_normalized.min():.3f}, max={X_normalized.max():.3f}")
print(f"\ny (xGoal) stats:")
print(f"  Mean: {y.mean():.4f}, Std: {y.std():.4f}")

## 3. Define Candidate Kernels


In [ ]:
# Kernel specifications
# Matérn kernels assume twice-differentiability; 5/2 is common for smooth functions
# RBF assumes infinite differentiability (often too smooth for real data)

kernels = {
    'Matérn 5/2': C(1.0) * Matern(length_scale=1.0, nu=2.5),
    'Matérn 3/2': C(1.0) * Matern(length_scale=1.0, nu=1.5),
    'RBF': C(1.0) * RBF(length_scale=1.0)
}

print("Candidate Kernels:")
for name, kernel in kernels.items():
    print(f"\n  {name}:")
    print(f"    {kernel}")
    print(f"    Interpretation: Matérn ν=2.5 is default for smooth, realistic data")
    print(f"    Hypothesis: Shot quality trajectory follows smooth curve")

## 4. Train & Evaluate Models


In [ ]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C
import matplotlib.pyplot as plt

# Create time windows (like Phase 1)
time_bins = [0, 300, 600, 900, 1200]
bin_labels = ['0–5 min', '5–10 min', '10–15 min', '15–20 min']

df = pd.DataFrame({
    'time': X_normalized.flatten() * 1200,  # Convert back to seconds
    'xGoal': y
})

# Aggregate by time window
aggregated = []
for i in range(len(time_bins)-1):
    mask = (df['time'] >= time_bins[i]) & (df['time'] < time_bins[i+1])
    mean_xgoal = df[mask]['xGoal'].mean()
    std_xgoal = df[mask]['xGoal'].std()
    n_shots = mask.sum()
    
    aggregated.append({
        'time': (time_bins[i] + time_bins[i+1]) / 2,
        'xGoal': mean_xgoal,
        'std': std_xgoal,
        'n': n_shots,
        'label': bin_labels[i]
    })

agg_df = pd.DataFrame(aggregated)
print("Aggregated data by time window:\n")
print(agg_df.to_string(index=False))

# Fit GP to aggregated data
X_agg = agg_df['time'].values.reshape(-1, 1) / 1200  # Normalize
y_agg = agg_df['xGoal'].values

print(f"\n\nFitting GP to {len(X_agg)} aggregated points...\n")

kernel = C(1.0) * Matern(length_scale=0.3, nu=2.5)

gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=0.01,
    normalize_y=True,
    n_restarts_optimizer=0,
    random_state=42
)

gp.fit(X_agg, y_agg)
train_r2 = gp.score(X_agg, y_agg)

print(f"Matérn 5/2 on aggregated data: R² = {train_r2:.4f}")

# Visualization
X_grid = np.linspace(0, 1, 100).reshape(-1, 1)
y_grid, y_std = gp.predict(X_grid, return_std=True)

plt.figure(figsize=(10, 5))
plt.scatter(X_agg, y_agg, s=100, color='steelblue', label='Aggregated mean xGoal', zorder=3)
plt.plot(X_grid, y_grid, 'r-', linewidth=2.5, label='GP posterior mean')
plt.fill_between(X_grid.flatten(), y_grid - 1.96*y_std, y_grid + 1.96*y_std,
                 alpha=0.2, color='red', label='95% Credible Band')
plt.xlabel('Time in Period (normalized)')
plt.ylabel('Mean xGoal')
plt.title(f'GP Fit to Aggregated Data (R² = {train_r2:.4f})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR /'gp_aggregated.png', dpi=150, bbox_inches='tight')
plt.show()

# Print posterior estimates
print(f"\nPosterior estimates by time window:")
for i, label in enumerate(bin_labels):
    print(f"  {label}: {y_agg[i]:.4f} xGoal")

print(f"\nIncrease from early to late: {(y_agg[-1]/y_agg[0] - 1)*100:.1f}%")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, RBF, ConstantKernel as C
import matplotlib.pyplot as plt

print("="*80)
print("PHASE 2 FINAL: KERNEL COMPARISON ON AGGREGATED DATA")
print("="*80)

# Aggregate data
time_bins = [0, 300, 600, 900, 1200]
bin_labels = ['0–5 min', '5–10 min', '10–15 min', '15–20 min']

df = pd.DataFrame({
    'time': X_normalized.flatten() * 1200,
    'xGoal': y
})

aggregated = []
for i in range(len(time_bins)-1):
    mask = (df['time'] >= time_bins[i]) & (df['time'] < time_bins[i+1])
    aggregated.append({
        'time': (time_bins[i] + time_bins[i+1]) / 2,
        'xGoal': df[mask]['xGoal'].mean(),
        'label': bin_labels[i]
    })

agg_df = pd.DataFrame(aggregated)
X_agg = agg_df['time'].values.reshape(-1, 1) / 1200
y_agg = agg_df['xGoal'].values

print("\nAggregated data:\n")
print(agg_df.to_string(index=False))

# Define kernels
kernels = {
    'Matérn 5/2': C(1.0) * Matern(length_scale=0.3, nu=2.5),
    'Matérn 3/2': C(1.0) * Matern(length_scale=0.3, nu=1.5),
    'RBF': C(1.0) * RBF(length_scale=0.3)
}

# Fit and compare
print("\n\nFitting kernels...\n")

results = {}
models = {}

for kernel_name, kernel in kernels.items():
    print(f"  {kernel_name}...", end=' ', flush=True)
    
    gp = GaussianProcessRegressor(
        kernel=kernel,
        alpha=0.01,
        normalize_y=True,
        n_restarts_optimizer=0,
        random_state=42
    )
    gp.fit(X_agg, y_agg)
    models[kernel_name] = gp
    
    r2 = gp.score(X_agg, y_agg)
    results[kernel_name] = {'r2': r2}
    
    print(f"✓ R²={r2:.4f}")

# Results
print("\n" + "="*80)
print("KERNEL COMPARISON RESULTS")
print("="*80 + "\n")

results_df = pd.DataFrame(results).T
print(results_df.to_string())

best_kernel = results_df['r2'].idxmax()
print(f"\n BEST KERNEL: {best_kernel}")
print(f"   R² = {results_df.loc[best_kernel, 'r2']:.4f}")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (kernel_name, ax) in enumerate(zip(kernels.keys(), axes)):
    gp = models[kernel_name]
    
    X_grid = np.linspace(0, 1, 200).reshape(-1, 1)
    y_grid, y_std = gp.predict(X_grid, return_std=True)
    X_grid_time = X_grid.flatten() * 1200
    
    # Plot
    ax.scatter(agg_df['time'], agg_df['xGoal'], s=150, color='steelblue', 
              edgecolor='black', linewidth=1.5, zorder=3)
    ax.plot(X_grid_time, y_grid, 'r-', linewidth=2.5, label='Posterior Mean')
    ax.fill_between(X_grid_time, y_grid - 1.96*y_std, y_grid + 1.96*y_std,
                    alpha=0.25, color='red')
    
    is_best = " 🏆 BEST" if kernel_name == best_kernel else ""
    ax.set_title(f'{kernel_name}\nR²={results[kernel_name]["r2"]:.4f}{is_best}', 
                fontweight='bold')
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('xGoal')
    ax.set_xlim(0, 1200)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR /'phase2_kernel_comparison.png', dpi=300, bbox_inches='tight')
print("\n Saved: phase2_kernel_comparison.png")
plt.show()

# Summary
print("\n" + "="*80)
print("POSTERIOR ESTIMATES")
print("="*80 + "\n")

for idx, row in agg_df.iterrows():
    print(f"{row['label']:12s}: {row['xGoal']:.4f} xGoal")

increase = (agg_df['xGoal'].iloc[-1] / agg_df['xGoal'].iloc[0] - 1) * 100
print(f"\nIncrease: {increase:.1f}%")

print("\n PHASE 2 COMPLETE!")

## 5. Posterior Predictions & Uncertainty


In [ ]:
# Use best model (Matérn 5/2 by default hypothesis)
best_kernel = 'Matérn 5/2'
best_gp = models[best_kernel]

# Create fine grid for predictions
X_test_normalized = np.linspace(0, 1, 200).reshape(-1, 1)
X_test = X_test_normalized * (X_max - X_min) + X_min  # Denormalize

# Predict mean and uncertainty
y_pred_test, y_std_test = best_gp.predict(X_test_normalized, return_std=True)

# Credible bands (95% confidence)
y_lower = y_pred_test - 1.96 * y_std_test
y_upper = y_pred_test + 1.96 * y_std_test

print(f"Posterior predictions generated for {best_kernel}")
print(f"  Mean xGoal at t=0s: {y_pred_test[0]:.4f}")
print(f"  Mean xGoal at t=1110s: {y_pred_test[-1]:.4f}")
print(f"  Difference: {y_pred_test[-1] - y_pred_test[0]:.4f} (absolute)")
print(f"  % Change: {100 * (y_pred_test[-1] - y_pred_test[0]) / y_pred_test[0]:.1f}%")

## 6. Visualization: Posterior Fit with Credible Bands


In [ ]:
# Main posterior plot
fig, ax = plt.subplots(figsize=(13, 7))

# 95% credible band
ax.fill_between(
    X_test.flatten(), y_lower, y_upper,
    alpha=0.25, color='steelblue', label='95% Credible Band'
)

# Posterior mean
ax.plot(X_test, y_pred_test, color='steelblue', linewidth=2.5, label='Posterior Mean')

# Raw data
ax.scatter(
    comeback_df['time'], comeback_df['xGoal'],
    alpha=0.3, s=25, color='darkgray', label='Observed Shots'
)

# Time window boundaries
for boundary in [300, 600, 900]:
    ax.axvline(boundary, color='red', linestyle='--', alpha=0.3, linewidth=1)

ax.set_xlabel('Time Elapsed in 3rd Period (seconds)', fontsize=12)
ax.set_ylabel('xGoal per Shot', fontsize=12)
ax.set_title(f'Gaussian Process Regression: Shot Quality Trajectory\n({best_kernel} Kernel with 95% Credible Bands)', 
             fontsize=13, fontweight='bold')
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(X_test.min(), X_test.max())

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'posterior_fit_credible_bands.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Posterior fit plot saved to {RESULTS_DIR}")

## 7. Model Comparison Visualization


In [ ]:
# Compare all three kernels
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for idx, (kernel_name, gp_model) in enumerate(models.items()):
    ax = axes[idx]
    
    # Predictions
    y_pred_k, y_std_k = gp_model.predict(X_test_normalized, return_std=True)
    y_lower_k = y_pred_k - 1.96 * y_std_k
    y_upper_k = y_pred_k + 1.96 * y_std_k
    
    # Plot
    ax.fill_between(X_test.flatten(), y_lower_k, y_upper_k, alpha=0.25, color='steelblue')
    ax.plot(X_test, y_pred_k, color='steelblue', linewidth=2.5)
    ax.scatter(comeback_df['time'], comeback_df['xGoal'], alpha=0.2, s=20, color='darkgray')
    
    # Metrics text
    rmse = cv_results[kernel_name]['rmse_cv']
    r2 = cv_results[kernel_name]['r2_train']
    ax.text(0.05, 0.95, f'CV RMSE: {rmse:.4f}\nR²: {r2:.4f}',
            transform=ax.transAxes, fontsize=10,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    ax.set_xlabel('Time (seconds)', fontsize=11)
    ax.set_title(kernel_name, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('xGoal per Shot', fontsize=11)
fig.suptitle('Kernel Comparison: Posterior Fits', fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig(RESULTS_DIR /'kernel_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Residual Diagnostics


In [ ]:
# In-sample residuals for best model
y_pred_in = best_gp.predict(X_normalized)
residuals = y - y_pred_in

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# Residuals vs. Fitted
axes[0, 0].scatter(y_pred_in, residuals, alpha=0.5, s=25)
axes[0, 0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0, 0].set_xlabel('Fitted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs. Fitted')
axes[0, 0].grid(True, alpha=0.3)

# Residuals vs. Time
axes[0, 1].scatter(comeback_df['time'], residuals, alpha=0.5, s=25, color='steelblue')
axes[0, 1].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0, 1].set_xlabel('Time (seconds)')
axes[0, 1].set_ylabel('Residuals')
axes[0, 1].set_title('Residuals vs. Time')
axes[0, 1].grid(True, alpha=0.3)

# Q-Q plot
from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot')
axes[1, 0].grid(True, alpha=0.3)

# Histogram of residuals
axes[1, 1].hist(residuals, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
axes[1, 1].set_xlabel('Residuals')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title(f'Residual Distribution (μ={residuals.mean():.4f}, σ={residuals.std():.4f})')

fig.suptitle(f'Model Diagnostics: {best_kernel}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR /'residual_diagnostics.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nResidual Statistics:")
print(f"  Mean: {residuals.mean():.6f} (should be ≈ 0)")
print(f"  Std: {residuals.std():.4f}")
print(f"  Min: {residuals.min():.4f}")
print(f"  Max: {residuals.max():.4f}")

## 9. Summary & Recommendations for Formalization


In [ ]:
print("\n" + "="*80)
print("EXPERIMENTATION PHASE SUMMARY")
print("="*80)

best_by_cv = cv_df['rmse_cv'].idxmin()
print(f"\nBest kernel by CV RMSE: {best_by_cv}")
print(f"  CV RMSE: {cv_df.loc[best_by_cv, 'rmse_cv']:.4f}")
print(f"  MAE (CV): {cv_df.loc[best_by_cv, 'mae_cv']:.4f}")
print(f"  R² (train): {cv_df.loc[best_by_cv, 'r2_train']:.4f}")

print(f"\nTemporal Trajectory (using {best_by_cv}):")
idx_0 = np.argmin(np.abs(X_test - X.min()))
idx_end = np.argmin(np.abs(X_test - X.max()))
xgoal_start = y_pred_test[idx_0]
xgoal_end = y_pred_test[idx_end]
print(f"  xGoal at 0 min: {xgoal_start:.4f}")
print(f"  xGoal at 18.5 min: {xgoal_end:.4f}")
print(f"  Change: {xgoal_end - xgoal_start:+.4f} ({100*(xgoal_end - xgoal_start)/xgoal_start:+.1f}%)")
print(f"  Direction: {'INCREASING' if xgoal_end > xgoal_start else 'DECREASING'}")

print(f"\nModel Diagnostics:")
print(f"  Residual mean: {residuals.mean():.6f} ✓")
print(f"  Approximate normality: Visual inspection via Q-Q plot")
print(f"  Homoscedasticity: Check residuals vs. time/fitted")

print(f"\n" + "="*80)
print(f"READY FOR PHASE 3: Formalization")
print(f"Recommendation: Use {best_by_cv} kernel for final model")
print(f"="*80)

## 10. Save Model for Next Phase


In [ ]:
import pickle

# Save the best model and data
model_artifacts = {
    'best_model': best_gp,
    'kernel_name': best_kernel,
    'X_normalized': X_normalized,
    'y': y,
    'X_test_normalized': X_test_normalized,
    'X_test': X_test,
    'y_pred_test': y_pred_test,
    'y_std_test': y_std_test,
    'cv_results': cv_results
}

with open( DATA_DIR / 'best_gp_model.pkl', 'wb') as f:
    pickle.dump(model_artifacts, f)

print(f"Model artifacts saved to { DATA_DIR / 'best_gp_model.pkl'}")
print(f"\nExperimentation phase complete. Ready for formalization.")